# Step 3:-  Converting to Embeddings and then storing in vector database

In [5]:
# Load the same documents and create the chunks needed for embedding.
# This keeps this notebook independently runnable for now.

from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Locate our PDF collection.
documents_path = Path("data/documents")
pdf_files = list(documents_path.glob("*.pdf"))

# Load all pages from all PDFs.
all_documents = []

for pdf_file in pdf_files:
    loader = PyPDFLoader(str(pdf_file))
    docs = loader.load()

    # Preserve the source filename for future citations.
    for doc in docs:
        doc.metadata["source_file"] = pdf_file.name

    all_documents.extend(docs)

# Split documents using the configuration we selected earlier.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(all_documents)

print("Total documents/pages:", len(all_documents))
print("Total chunks:", len(chunks))

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_5988\2962571149.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Total documents/pages: 77
Total chunks: 481


In [6]:
# HuggingFaceEmbeddings provides an interface to HuggingFace
# sentence-transformer models for generating text embeddings.

from langchain_huggingface import HuggingFaceEmbeddings

In [7]:
# We use a lightweight and widely used sentence-transformer model.
# It converts text into numerical vectors that capture semantic meaning.

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3218.55it/s]


Embedding model loaded successfully.


In [8]:
# Take one chunk from our document collection.
sample_text = chunks[0].page_content

# Convert the text into a numerical vector.
sample_embedding = embedding_model.embed_query(sample_text)

print("Embedding type:", type(sample_embedding))
print("Embedding dimensions:", len(sample_embedding))
print("First 10 values:", sample_embedding[:10])

Embedding type: <class 'list'>
Embedding dimensions: 384
First 10 values: [-0.11447722464799881, -0.12833653390407562, 0.03361820429563522, -0.014195232652127743, 0.042737822979688644, 0.06814958155155182, -0.07230935245752335, -0.006441767327487469, 0.11104848235845566, -0.020435139536857605]


In [9]:
# The embedding vector is a list of numbers.
# Each dimension represents one learned feature of the text.

print("Number of dimensions:", len(sample_embedding))

# Display the complete vector only if you want to inspect it.
# For a 384-dimensional model, this will be a long list.
print("First 20 dimensions:", sample_embedding[:20])

Number of dimensions: 384
First 20 dimensions: [-0.11447722464799881, -0.12833653390407562, 0.03361820429563522, -0.014195232652127743, 0.042737822979688644, 0.06814958155155182, -0.07230935245752335, -0.006441767327487469, 0.11104848235845566, -0.020435139536857605, 0.021149028092622757, -0.05593105033040047, -0.058749858289957047, 0.04953864961862564, -0.09599779546260834, 0.02417960949242115, -0.003824967425316572, 0.03941744565963745, -0.06117541715502739, -0.08560001105070114]


In [10]:
# Embeddings should place semantically similar text closer together
# than unrelated text.

text_1 = "The Transformer architecture uses self-attention."
text_2 = "Self-attention is a key component of Transformers."
text_3 = "The weather is sunny today."

embedding_1 = embedding_model.embed_query(text_1)
embedding_2 = embedding_model.embed_query(text_2)
embedding_3 = embedding_model.embed_query(text_3)

print("All three embeddings generated successfully.")

All three embeddings generated successfully.


In [11]:
# Cosine similarity measures how similar two vectors are.
# A value closer to 1 generally means the vectors point in a
# more similar direction.

from sklearn.metrics.pairwise import cosine_similarity

similarity_12 = cosine_similarity(
    [embedding_1],
    [embedding_2]
)[0][0]

similarity_13 = cosine_similarity(
    [embedding_1],
    [embedding_3]
)[0][0]

print("Similarity between Transformer sentences:", similarity_12)
print("Similarity between Transformer and weather:", similarity_13)

Similarity between Transformer sentences: 0.8306892561356566
Similarity between Transformer and weather: -0.02764809162021107


# Vector Database Storing 

In [13]:
# Chroma is our vector database.
# It will store our document chunks along with their embeddings
# and allow us to perform semantic similarity searches.

from langchain_chroma import Chroma

print("ChromaDB integration imported successfully.")

ChromaDB integration imported successfully.


In [14]:
# Create a Chroma vector store from our document chunks.
#
# Chroma will:
# 1. Take each chunk
# 2. Generate its embedding using embedding_model
# 3. Store the chunk + embedding + metadata
#
# persist_directory makes the database persistent on disk.

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="chroma_db"
)

print("ChromaDB created successfully.")
print("Total chunks stored:", len(chunks))

ChromaDB created successfully.
Total chunks stored: 481


In [15]:
# Now let's search our vector database using a natural-language question.
#
# Chroma will convert the query into an embedding and compare it
# with the stored document embeddings.

query = "How does the Transformer architecture use attention?"

results = vectorstore.similarity_search(
    query,
    k=3
)

print("Number of results:", len(results))

for i, result in enumerate(results):
    print(f"\n{'=' * 60}")
    print(f"RESULT {i + 1}")
    print(f"{'=' * 60}")
    
    print(result.page_content[:500])
    print("\nSource:", result.metadata.get("source_file"))
    print("Page:", result.metadata.get("page"))

Number of results: 3

RESULT 1
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence models such as
[38, 2, 9].
• The encoder contains self-attention layers. In a self-attention la

Source: Attention Is All You Need.pdf
Page: 4

RESULT 2
collected through web sources. This data contains private
information; therefore, many LLMs employ heuristics-based
methods to filter information such as names, addresses, and
phone numbers to avoid learning personal information.
2.9. Architectures
Here we discuss the variants of the transformer architectures
used in LLMs. The di fference arises due to the application of
Figure 4: An example of attention pat